# Assignment 2: Advanced NLP – IMDB Sentiment Analysis
Building on Assignment 1 (MLP + VADER/TextBlob), we now train:
1. **Optimized MLP** – Adam, Dropout, BatchNorm, full embeddings
2. **Bidirectional LSTM**
3. **Bidirectional GRU**
4. **1D CNN (TextCNN)**
5. **Word2Vec Embeddings + BiLSTM** (Gensim)

In [ ]:
!pip install gensim -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from gensim.models import Word2Vec

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow {tf.__version__}')
print('All libraries imported!')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('IMDB_Dataset.csv')
print(f'Shape: {df.shape}')
print(df['sentiment'].value_counts())
df = df.drop_duplicates().reset_index(drop=True)
print(f'After dedup: {df.shape[0]} rows')

## 2. Text Preprocessing

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', '', text)       # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)      # letters only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Cleaning reviews...')
df['clean_review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].map({'negative': 0, 'positive': 1})

# Token lists (used later by Word2Vec)
df['tokens'] = df['clean_review'].apply(
    lambda t: [w for w in t.split() if w not in STOP_WORDS and len(w) > 1]
)
print('Done!')
print(f"Sample: {df['tokens'].iloc[0][:10]}")

## 3. Train/Val/Test Split & Keras Tokenization

In [ ]:
VOCAB_SIZE  = 20000
MAX_LEN     = 200
EMBED_DIM   = 64
BATCH_SIZE  = 256
EPOCHS      = 10

texts  = df['clean_review'].values
labels = df['label'].values

# 80 / 10 / 10 split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

print(f'Train: {len(X_tr)} | Val: {len(X_val)} | Test: {len(X_te)}')

# Fit tokenizer on TRAIN only (no data leakage)
tok = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tok.fit_on_texts(X_tr)

def encode(texts):
    seqs = tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_tr_seq  = encode(X_tr)
X_val_seq = encode(X_val)
X_te_seq  = encode(X_te)

print(f'X_train shape: {X_tr_seq.shape}')

## 4. Shared Helper Functions

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=3,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=2, min_lr=1e-6, verbose=1),
    ]

def evaluate_model(model, X_test, y_test, name):
    probs = model.predict(X_test, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)
    acc   = accuracy_score(y_test, preds) * 100
    print(f'\n{'='*50}')
    print(f' {name}  —  Test Accuracy: {acc:.2f}%')
    print(f'{'='*50}')
    print(classification_report(y_test, preds,
          target_names=['Negative','Positive']))
    return acc

def plot_history(h, name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h.history['loss'],     label='Train')
    axes[0].plot(h.history['val_loss'], label='Val')
    axes[0].set_title(f'{name} — Loss');   axes[0].legend()
    axes[1].plot(h.history['accuracy'],     label='Train')
    axes[1].plot(h.history['val_accuracy'], label='Val')
    axes[1].set_title(f'{name} — Accuracy'); axes[1].legend()
    plt.tight_layout(); plt.show()

results = {}   # store final test accuracies
print('Helpers ready!')

## Model 1 — Optimized MLP
**Key improvements over Assignment 1:**
- Full text sequences via an Embedding layer (vs. 2 polarity scores)
- Adam optimizer (vs. vanilla gradient descent)
- Dropout + Batch Normalization
- EarlyStopping & ReduceLROnPlateau callbacks

In [ ]:
def build_mlp():
    model = keras.Sequential([
        layers.Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
        layers.GlobalAveragePooling1D(),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ], name='Optimized_MLP')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

mlp = build_mlp()
mlp.summary()

mlp_h = mlp.fit(X_tr_seq, y_tr,
                validation_data=(X_val_seq, y_val),
                epochs=EPOCHS, batch_size=BATCH_SIZE,
                callbacks=get_callbacks(), verbose=1)

plot_history(mlp_h, 'Optimized MLP')
results['Optimized MLP'] = evaluate_model(mlp, X_te_seq, y_te, 'Optimized MLP')

## Model 2 — Bidirectional LSTM
Reads the review forwards **and** backwards — great for capturing
long-range context like negation and contrast.

In [ ]:
def build_bilstm():
    model = keras.Sequential([
        layers.Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
        layers.SpatialDropout1D(0.2),
        layers.Bidirectional(
            layers.LSTM(64, return_sequences=True,
                        dropout=0.3, recurrent_dropout=0.2)),
        layers.Bidirectional(
            layers.LSTM(32, dropout=0.3, recurrent_dropout=0.2)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ], name='BiLSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm = build_bilstm()
lstm.summary()

lstm_h = lstm.fit(X_tr_seq, y_tr,
                  validation_data=(X_val_seq, y_val),
                  epochs=EPOCHS, batch_size=BATCH_SIZE,
                  callbacks=get_callbacks(), verbose=1)

plot_history(lstm_h, 'BiLSTM')
results['BiLSTM'] = evaluate_model(lstm, X_te_seq, y_te, 'Bidirectional LSTM')

## Model 3 — Bidirectional GRU
Same idea as LSTM but lighter (fewer parameters) → trains faster,
often reaches similar accuracy.

In [ ]:
def build_bigru():
    model = keras.Sequential([
        layers.Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
        layers.SpatialDropout1D(0.2),
        layers.Bidirectional(
            layers.GRU(64, return_sequences=True,
                       dropout=0.3, recurrent_dropout=0.2)),
        layers.Bidirectional(
            layers.GRU(32, dropout=0.3, recurrent_dropout=0.2)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ], name='BiGRU')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

gru = build_bigru()
gru.summary()

gru_h = gru.fit(X_tr_seq, y_tr,
                validation_data=(X_val_seq, y_val),
                epochs=EPOCHS, batch_size=BATCH_SIZE,
                callbacks=get_callbacks(), verbose=1)

plot_history(gru_h, 'BiGRU')
results['BiGRU'] = evaluate_model(gru, X_te_seq, y_te, 'Bidirectional GRU')

## Model 4 — 1D CNN (TextCNN)
Parallel Conv1D filters with kernel sizes 2–5 capture different
n-gram patterns simultaneously (e.g. 'not good', 'absolutely terrible').

In [ ]:
def build_textcnn():
    inp = keras.Input(shape=(MAX_LEN,))
    x   = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(inp)

    # Parallel convolutions — one branch per kernel size
    branches = []
    for k in [2, 3, 4, 5]:
        c = layers.Conv1D(64, k, activation='relu', padding='same')(x)
        p = layers.GlobalMaxPooling1D()(c)
        branches.append(p)

    x = layers.concatenate(branches)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64,  activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inp, out, name='TextCNN')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

cnn = build_textcnn()
cnn.summary()

cnn_h = cnn.fit(X_tr_seq, y_tr,
                validation_data=(X_val_seq, y_val),
                epochs=EPOCHS, batch_size=BATCH_SIZE,
                callbacks=get_callbacks(), verbose=1)

plot_history(cnn_h, '1D CNN')
results['1D CNN'] = evaluate_model(cnn, X_te_seq, y_te, '1D CNN (TextCNN)')

## Model 5 — Word2Vec Embeddings + BiLSTM
### Step 1: Train Word2Vec on the training corpus (Gensim)

In [ ]:
# Build token lists from TRAINING reviews only
train_token_lists = [t.split() for t in X_tr]

W2V_DIM = 64

print('Training Word2Vec...')
w2v = Word2Vec(
    sentences=train_token_lists,
    vector_size=W2V_DIM,
    window=5,
    min_count=2,
    workers=4,
    epochs=5,
    sg=0          # 0 = CBOW, 1 = Skip-Gram
)

print(f'W2V vocabulary: {len(w2v.wv)} words')
print('Similar to good:', w2v.wv.most_similar('good', topn=5))

### Step 2: Build the Embedding Matrix

In [ ]:
word_index   = tok.word_index
vocab_actual = min(VOCAB_SIZE, len(word_index) + 1)

emb_matrix = np.zeros((vocab_actual, W2V_DIM))
hits = misses = 0

for word, idx in word_index.items():
    if idx >= vocab_actual:
        continue
    if word in w2v.wv:
        emb_matrix[idx] = w2v.wv[word]
        hits += 1
    else:
        misses += 1

print(f'Embedding matrix: {emb_matrix.shape}')
print(f'Coverage: {hits} hits / {misses} misses '
      f'({hits/(hits+misses)*100:.1f}%)')

### Step 3: Build & Train Word2Vec + BiLSTM

In [ ]:
def build_w2v_bilstm(emb_matrix, vocab_size, embed_dim):
    model = keras.Sequential([
        # Embedding initialized from Word2Vec, fine-tuned during training
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            weights=[emb_matrix],
            input_length=MAX_LEN,
            trainable=True
        ),
        layers.SpatialDropout1D(0.2),
        layers.Bidirectional(
            layers.LSTM(64, return_sequences=True,
                        dropout=0.3, recurrent_dropout=0.2)),
        layers.Bidirectional(
            layers.LSTM(32, dropout=0.3, recurrent_dropout=0.2)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ], name='W2V_BiLSTM')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

w2v_lstm = build_w2v_bilstm(emb_matrix, vocab_actual, W2V_DIM)
w2v_lstm.summary()

w2v_h = w2v_lstm.fit(X_tr_seq, y_tr,
                     validation_data=(X_val_seq, y_val),
                     epochs=EPOCHS, batch_size=BATCH_SIZE,
                     callbacks=get_callbacks(), verbose=1)

plot_history(w2v_h, 'Word2Vec + BiLSTM')
results['Word2Vec + BiLSTM'] = evaluate_model(
    w2v_lstm, X_te_seq, y_te, 'Word2Vec + BiLSTM')

## Final Comparison

In [ ]:
# ── Replace 70.5 with your actual Assignment 1 test accuracy ──
all_results = {'Asgn 1 MLP (VADER+TextBlob)': 70.5}
all_results.update(results)

print(f"{'Model':<35} {'Test Acc':>10}")
print('-' * 47)
for name, acc in sorted(all_results.items(), key=lambda x: x[1], reverse=True):
    print(f'{name:<35} {acc:>9.2f}%')

# Bar chart
plt.figure(figsize=(10, 5))
colors = ['#888888'] + ['#2196F3'] * len(results)
names  = list(all_results.keys())
accs   = list(all_results.values())
bars = plt.barh(names, accs, color=colors)
plt.xlabel('Test Accuracy (%)')
plt.title('Assignment 1 vs Assignment 2 — All Models')
plt.xlim(60, 100)
for bar, acc in zip(bars, accs):
    plt.text(acc + 0.2, bar.get_y() + bar.get_height()/2,
             f'{acc:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Save Best Model & Generate Submission CSV

In [ ]:
model_map = {
    'Optimized MLP':    mlp,
    'BiLSTM':           lstm,
    'BiGRU':            gru,
    '1D CNN':           cnn,
    'Word2Vec + BiLSTM': w2v_lstm,
}

best_name  = max(results, key=results.get)
best_model = model_map[best_name]
print(f'Best model: {best_name}  ({results[best_name]:.2f}%)')

best_model.save('best_model_assignment2.h5')
print('Saved: best_model_assignment2.h5')

probs  = best_model.predict(X_te_seq, verbose=0).flatten()
preds  = (probs >= 0.5).astype(int)

sub = pd.DataFrame({'id': range(len(preds)),
                    'sentiment_pred': preds,
                    'probability': probs.round(4)})
sub.to_csv('assignment2_submission.csv', index=False)
print('Saved: assignment2_submission.csv')
print(sub.head())